## Part 1 — Setup & launch


In [1]:
# ── 1. GPU check ────────────────────────────────────────────────────────────
# Confirms the runtime has an NVIDIA GPU. Everything still works on CPU, but
# a single sentence can take minutes instead of seconds.
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("GPU detected — you're good to go.")
else:
    print("=" * 74)
    print("WARNING: no NVIDIA GPU in this runtime.")
    print("Fix: menu bar -> Runtime -> Change runtime type -> T4 GPU,")
    print("then re-run this notebook from the top.")
    print("(Continuing anyway works, but generation will be very slow on CPU.)")
    print("=" * 74)


Fri Oct  9 18:31:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 2

In [2]:
# ── 2. Clone/reuse + install dependencies + build frontend ────────────────

import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/debpalash/VoiceStudio"

REPO_DIR = Path("/kaggle/working/VoiceStudio")
BUN_INSTALL = Path("/kaggle/working/.bun")
PERSISTENT_BUN = BUN_INSTALL / "bin" / "bun"

# Execute Bun from the runtime filesystem, not directly from persistent storage
BUN = Path("/usr/local/bin/bun")
os.environ["BUN_INSTALL"] = str(BUN_INSTALL)
os.environ["UV_CACHE_DIR"] = "/kaggle/working/.uv-cache"
os.environ["PIP_CACHE_DIR"] = "/kaggle/working/.pip-cache"

os.environ["PATH"] = (
    str(BUN_INSTALL / "bin")
    + os.pathsep
    + "/kaggle/working/bin"
    + os.pathsep
    + os.environ.get("PATH", "")
)

os.environ.setdefault("UV_HTTP_TIMEOUT", "300")


def run(cmd, *, cwd=None, what=""):
    shown = cmd if isinstance(cmd, str) else " ".join(map(str, cmd))
    print(f"\n$ {shown}")

    env = os.environ.copy()

    bun_dir = str(BUN.parent)

    if bun_dir not in env["PATH"]:
        env["PATH"] = bun_dir + os.pathsep + env["PATH"]

    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        shell=isinstance(cmd, str),
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"\nFAILED: {what or shown}\n"
            f"Exit code: {result.returncode}\n"
        )


# -------------------------------------------------------------------
# 2a. Clone repository only if it does not already exist
# -------------------------------------------------------------------

if (REPO_DIR / ".git").is_dir():
    print(f"✅ Repo already exists: {REPO_DIR}")
    print("Reusing persisted VoiceStudio repository.")
else:
    print("⬇️ Cloning VoiceStudio...")
    run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        what="git clone",
    )


# -------------------------------------------------------------------
# 2b. System packages
# -------------------------------------------------------------------

if shutil.which("ffmpeg"):
    print("✅ ffmpeg already installed.")
else:
    run(
        "apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1",
        what="system packages",
    )


# ── 2c. Restore Bun from persistent storage into the runtime ────────────────

import shutil
import subprocess
from pathlib import Path

BUN_INSTALL = Path("/kaggle/working/.bun")
PERSISTENT_BUN = BUN_INSTALL / "bin" / "bun"

# Download Bun only if the persisted binary is missing.
if not PERSISTENT_BUN.is_file():
    print("Persistent Bun binary is missing. Installing Bun...")

    BUN_INSTALL.mkdir(parents=True, exist_ok=True)

    install_env = os.environ.copy()
    install_env["BUN_INSTALL"] = str(BUN_INSTALL)

    subprocess.run(
        ["bash", "-c", "curl -fsSL https://bun.sh/install | bash"],
        env=install_env,
        check=True,
    )

if not PERSISTENT_BUN.is_file():
    raise RuntimeError(f"Bun installer did not create {PERSISTENT_BUN}")

# Copy Bun into a normal runtime executable location and restore permissions.
# Fall back to /tmp if /usr/local/bin is not writable or executable.
BUN = None
errors = []

for candidate in [
    Path("/usr/local/bin/bun"),
    Path("/tmp/bun"),
]:
    try:
        shutil.copyfile(PERSISTENT_BUN, candidate)
        os.chmod(candidate, 0o755)

        result = subprocess.run(
            [str(candidate), "--version"],
            capture_output=True,
            text=True,
            check=True,
        )

        BUN = candidate
        print("✅ Bun executable restored.")
        print("Runtime path:", BUN)
        print("Bun version:", result.stdout.strip())
        break

    except (OSError, subprocess.CalledProcessError) as exc:
        errors.append(f"{candidate}: {exc}")
        print(f"Could not use {candidate}; trying fallback.")

if BUN is None:
    raise RuntimeError(
        "Could not execute Bun.\n" + "\n".join(errors)
    )

# Make the working runtime's Bun available to package scripts.
os.environ["PATH"] = (
    str(BUN.parent)
    + os.pathsep
    + "/kaggle/working/bin"
    + os.pathsep
    + os.environ.get("PATH", "")
)


# -------------------------------------------------------------------
# 2d. Frontend
# -------------------------------------------------------------------

frontend_dist = REPO_DIR / "frontend" / "dist"
dist_index = frontend_dist / "index.html"

# Remove the WRONG symlink created by the previous Gemini workaround.
if frontend_dist.is_symlink():
    print("⚠️ Removing old frontend/dist symlink...")
    frontend_dist.unlink()

# If an old incorrect out/renderer build exists, leave it alone.
# The official web build will create frontend/dist itself.

if dist_index.exists():
    print("✅ frontend/dist/index.html already exists.")
    print("Skipping frontend build.")
else:

    node_modules = REPO_DIR / "node_modules"

    if node_modules.exists():
        print("✅ Persisted node_modules found.")
        print("Skipping Bun dependency download.")
    else:
        print("⬇️ Installing JavaScript dependencies...")
        run(
            [str(BUN), "install", "--frozen-lockfile"],
            cwd=REPO_DIR,
            what="bun install",
        )

    print("\n🏗️ Building VoiceStudio web frontend...")
    print("Using the official web build command.")

    run(
        [str(BUN), "run", "build:web"],
        cwd=REPO_DIR,
        what="VoiceStudio web frontend build",
    )

    if not dist_index.exists():
        raise RuntimeError(
            "Frontend build completed but frontend/dist/index.html "
            "was not created."
        )

    print("✅ Frontend build complete.")


# -------------------------------------------------------------------
# 2e. Backend Python dependencies
# -------------------------------------------------------------------

if shutil.which("uv"):
    print("✅ uv already available.")
else:
    print("⬇️ Installing uv...")
    run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "uv",
        ],
        what="pip install uv",
    )


print("\n📦 Installing/checking VoiceStudio Python dependencies...")

run(
    [
        "uv",
        "pip",
        "install",
        "--system",
        "--constraint",
        "deploy/torch-constraints.txt",
        ".",
    ],
    cwd=REPO_DIR,
    what="VoiceStudio backend dependencies",
)


# -------------------------------------------------------------------
# 2f. cuDNN compatibility
# -------------------------------------------------------------------

venv_dir = REPO_DIR / ".venv"
venv_dir.mkdir(exist_ok=True)

run(
    [
        sys.executable,
        str(REPO_DIR / "scripts" / "setup.py"),
    ],
    what="cuDNN compatibility setup",
)


# -------------------------------------------------------------------
# 2g. Import sanity check
# -------------------------------------------------------------------

print("\n🔍 Running model-stack sanity check...")

run(
    [
        sys.executable,
        "-c",
        (
            "import torch, torchaudio, uvicorn, fastapi, transformers; "
            "print("
            "f'torch {torch.__version__}, "
            "torchaudio {torchaudio.__version__}, "
            "transformers {transformers.__version__}, "
            "CUDA available: {torch.cuda.is_available()}'"
            "); "
            "from omnivoice.models.omnivoice import OmniVoice; "
            "from transformers import HiggsAudioV2TokenizerModel; "
            "print('✅ VoiceStudio model stack imports cleanly')"
        ),
    ],
    cwd=REPO_DIR,
    what="model-stack sanity check",
)

print("\n" + "=" * 80)
print("✅ CELL 2 COMPLETE")
print("=" * 80)

✅ Repo already exists: /kaggle/working/VoiceStudio
Reusing persisted VoiceStudio repository.
✅ ffmpeg already installed.
✅ Bun executable restored.
Runtime path: /usr/local/bin/bun
Bun version: 1.4.2
✅ frontend/dist/index.html already exists.
Skipping frontend build.
✅ uv already available.

📦 Installing/checking VoiceStudio Python dependencies...

$ uv pip install --system --constraint deploy/torch-constraints.txt .


Using Python 3.13.15 environment at: /usr
Resolved 239 packages in 2.25s
Prepared 79 packages in 26.45s
Uninstalled 17 packages in 3.18s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 79 packages in 37.18s
 + addict==2.4.0
 + argostranslate==1.11.0
 + asteroid-filterbanks==0.4.0
 + audioseal==0.2.0
 + av==15.1.0
 + backoff==2.2.1
 - boto3==1.41.5
 + boto3==1.43.110
 - botocore==1.41.5
 + botocore==1.43.110
 + braceexpand==0.1.7
 + csvw==4.1.0
 + ctranslate2==4.8.2
 + curated-tokenizers==0.0.10
 + curated-transformers==0.1.1
 + demucs==4.1.0
 + dlinfo==2.0.0
 + docopt==0.6.2
 + espeakng-loader==0.2.4
 - fastapi==0.141.1
 + fastapi==0.136.3
 + faster-whisper==1.2.1
 + httpx-sse==0.4.3
 + hyperpyyaml==1.2.3
 + isodate==0.7.2
 + julius==0.2.8
 + kittentts==0.8.1 (from https://github.com/KittenML/KittenTTS


$ /usr/bin/python3 /kaggle/working/VoiceStudio/scripts/setup.py
✓ cuDNN 8 compat: 7 libraries ready

🔍 Running model-stack sanity check...

$ /usr/bin/python3 -c import torch, torchaudio, uvicorn, fastapi, transformers; print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, transformers {transformers.__version__}, CUDA available: {torch.cuda.is_available()}'); from omnivoice.models.omnivoice import OmniVoice; from transformers import HiggsAudioV2TokenizerModel; print('✅ VoiceStudio model stack imports cleanly')


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


torch 2.8.0+cu128, torchaudio 2.8.0+cu128, transformers 5.16.1, CUDA available: True
✅ VoiceStudio model stack imports cleanly

✅ CELL 2 COMPLETE


# 3

In [3]:
# ── 3. Optional Hugging Face token ─────────────────────────────────────────

import os

try:
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()
    token = client.get_secret("HF_TOKEN")

    os.environ["HF_TOKEN"] = token

    print("✅ HF_TOKEN loaded from Kaggle Secrets.")
    print("Gated Hugging Face models are available.")

except Exception:
    print(
        "ℹ️ No HF_TOKEN found.\n"
        "This is fine for OmniVoice TTS, voice design and voice cloning.\n"
        "Some gated diarization/video-dubbing models may require a token."
    )

ℹ️ No HF_TOKEN found.
This is fine for OmniVoice TTS, voice design and voice cloning.
Some gated diarization/video-dubbing models may require a token.


# 4

In [4]:
# ── 4. Persistent OmniVoice model download/cache ───────────────────────────

import os
from pathlib import Path

# Make absolutely sure the persistent cache is active
CACHE_DIR = Path("/kaggle/working/omnivoice_cache")

os.environ["OMNIVOICE_CACHE_DIR"] = str(CACHE_DIR)
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR / "hub")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_DIR / "hub")
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")

CACHE_DIR.mkdir(parents=True, exist_ok=True)

from huggingface_hub import snapshot_download

print("=" * 80)
print("OMNIVOICE MODEL CACHE")
print("=" * 80)

print("Cache location:")
print(CACHE_DIR)

print("\nDownloading only if the model is missing...")
print("Already cached files will be reused automatically.\n")

model_path = snapshot_download(
    repo_id="k2-fsa/OmniVoice",
    cache_dir=os.environ["HF_HUB_CACHE"],
)

print("\n✅ OmniVoice model is available.")
print("Snapshot:", model_path)
print("Persistent cache:", CACHE_DIR)

OMNIVOICE MODEL CACHE
Cache location:
/kaggle/working/omnivoice_cache

Already cached files will be reused automatically.



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]


✅ OmniVoice model is available.
Snapshot: /kaggle/working/omnivoice_cache/hub/models--k2-fsa--OmniVoice/snapshots/c5fdb5ccb189668d56333f77ba2629f4cd7535f4
Persistent cache: /kaggle/working/omnivoice_cache


# New cell: Generate API key

In [5]:
# ── VoiceStudio remote-access API key ──
import secrets

# Keep the same key if this cell is accidentally rerun.
if not globals().get("VOICESTUDIO_API_KEY"):
    VOICESTUDIO_API_KEY = secrets.token_urlsafe(32)

print("✅ VoiceStudio API key created.")
print("It will be displayed next to your temporary URL.")

✅ VoiceStudio API key created.
It will be displayed next to your temporary URL.


# 5

In [6]:
# ── 5. Launch VoiceStudio backend ──────────────────────────────────────────

import json
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

REPO_DIR = Path("/kaggle/working/VoiceStudio")

PORT = 3900

DATA_DIR = Path("/kaggle/working/omnivoice_data")
CACHE_DIR = Path("/kaggle/working/omnivoice_cache")
LOG_PATH = Path("/kaggle/working/omnivoice_backend.log")

DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

HEALTH_URL = f"http://127.0.0.1:{PORT}/health"


def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as response:
            return json.load(response)
    except Exception:
        return None


print("=" * 80)
print("STARTING VOICESTUDIO BACKEND")
print("=" * 80)


# -------------------------------------------------------------------
# Kill old backend if this cell was previously run
# -------------------------------------------------------------------

print("Checking for existing backend...")

subprocess.run(
    "kill -9 $(lsof -t -i:3900) 2>/dev/null || true",
    shell=True,
)

time.sleep(2)


# -------------------------------------------------------------------
# Environment passed to backend
# -------------------------------------------------------------------

env = os.environ.copy()

# Headless web/server mode
env["OMNIVOICE_SERVER_MODE"] = "1"

# api key for temp link
env["OMNIVOICE_API_KEY"] = VOICESTUDIO_API_KEY

# PERSISTENT APP DATA
env["OMNIVOICE_DATA_DIR"] = str(DATA_DIR)

# PERSISTENT MODEL CACHE
env["OMNIVOICE_CACHE_DIR"] = str(CACHE_DIR)

# Hugging Face cache
env["HF_HOME"] = str(CACHE_DIR)
env["HF_HUB_CACHE"] = str(CACHE_DIR / "hub")
env["HUGGINGFACE_HUB_CACHE"] = str(CACHE_DIR / "hub")

# Torch cache
env["TORCH_HOME"] = str(CACHE_DIR / "torch")

env["PYTHONUNBUFFERED"] = "1"


# -------------------------------------------------------------------
# Start backend
# -------------------------------------------------------------------

log_file = open(LOG_PATH, "ab")

proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--app-dir",
        "backend",
        "--host",
        "127.0.0.1",
        "--port",
        str(PORT),
    ],
    cwd=REPO_DIR,
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print(f"Backend PID: {proc.pid}")
print(f"Backend log: {LOG_PATH}")
print(f"Persistent data: {DATA_DIR}")
print(f"Persistent model cache: {CACHE_DIR}")


# -------------------------------------------------------------------
# Wait for health
# -------------------------------------------------------------------

print("\nWaiting for VoiceStudio...")

deadline = time.time() + 300

while time.time() < deadline:

    if proc.poll() is not None:
        break

    info = health()

    if info:
        break

    print(".", end="", flush=True)
    time.sleep(3)

print()


# -------------------------------------------------------------------
# Final check
# -------------------------------------------------------------------

info = health()

if info:

    print("\n" + "=" * 80)
    print("✅ VOICESTUDIO BACKEND IS UP")
    print("=" * 80)

    print(json.dumps(info, indent=2))

    print("\nPersistent app data:")
    print(DATA_DIR)

    print("\nPersistent model cache:")
    print(CACHE_DIR)

else:

    try:
        with open(LOG_PATH, "r", errors="replace") as f:
            lines = f.readlines()

        tail = "".join(lines[-80:])

    except Exception:
        tail = "(unable to read backend log)"

    raise RuntimeError(
        "\nVoiceStudio backend did not become healthy.\n\n"
        f"--- last backend log lines ---\n"
        f"{tail}\n"
        f"--- end log ---"
    )

STARTING VOICESTUDIO BACKEND
Checking for existing backend...
Backend PID: 220
Backend log: /kaggle/working/omnivoice_backend.log
Persistent data: /kaggle/working/omnivoice_data
Persistent model cache: /kaggle/working/omnivoice_cache

Waiting for VoiceStudio...
...

✅ VOICESTUDIO BACKEND IS UP
{
  "status": "ok",
  "device": "cuda (Tesla T4)",
  "version": "0.5.7"
}

Persistent app data:
/kaggle/working/omnivoice_data

Persistent model cache:
/kaggle/working/omnivoice_cache


# 6

In [9]:
# ── 6. Open the VoiceStudio web UI through a temporary public tunnel ────────

import os
import re
import subprocess
import time
from pathlib import Path

PORT = 3900

BIN_DIR = Path("/kaggle/working/bin")
BIN_DIR.mkdir(parents=True, exist_ok=True)

CLOUDFLARED = BIN_DIR / "cloudflared"
LOG_PATH = Path("/kaggle/working/cloudflared.log")


# -------------------------------------------------------------------
# Install cloudflared ONCE into persistent storage
# -------------------------------------------------------------------

if CLOUDFLARED.exists():

    print("✅ Persistent cloudflared binary already exists.")

else:

    print("⬇️ Downloading cloudflared...")

    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O",
            str(CLOUDFLARED),
        ],
        check=True,
    )

    CLOUDFLARED.chmod(0o755)

    print("✅ cloudflared installed.")


# -------------------------------------------------------------------
# Stop an old tunnel if the cell was rerun
# -------------------------------------------------------------------

subprocess.run(
    ["pkill", "-f", "cloudflared tunnel --url"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(2)


# -------------------------------------------------------------------
# Start tunnel
# -------------------------------------------------------------------

print("\n🌐 Starting Cloudflare Quick Tunnel...")

log_file = open(LOG_PATH, "w")

import os
os.chmod(
    str(CLOUDFLARED),
    os.stat(str(CLOUDFLARED)).st_mode | 0o111
)

print("✅ Cloudflared execution permissions fixed.")

tunnel_proc = subprocess.Popen(
    [
        str(CLOUDFLARED),
        "tunnel",
        "--url",
        f"http://127.0.0.1:{PORT}",
        "--no-autoupdate",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

# -------------------------------------------------------------------
# Wait for public URL
# -------------------------------------------------------------------

public_url = None

for _ in range(30):

    if LOG_PATH.exists():

        text = LOG_PATH.read_text(
            encoding="utf-8",
            errors="replace",
        )

        matches = re.findall(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            text,
        )

        if matches:
            public_url = matches[-1]
            break

    time.sleep(2)


if not public_url:

    print("\n❌ Cloudflare did not produce a URL.")
    print("Check:")
    print(LOG_PATH.read_text(errors="replace"))
    raise RuntimeError("Cloudflare tunnel failed.")

print("\n" + "=" * 80)
print("🔥 VOICESTUDIO IS LIVE")
print("=" * 80)

print("\nOpen this URL:")
print(public_url)

print("\nYour VoiceStudio UI is now:")
print("Kaggle T4 → VoiceStudio backend → Cloudflare → your browser")

print("\n⚠️ Treat this URL as private.")
print("Use VoiceStudio's share/security settings before sharing it.")

# Save current URL for convenience
Path("/kaggle/working/active_voice_studio_url.txt").write_text(
    public_url,
    encoding="utf-8",
)

print("\nURL saved to:")
print("/kaggle/working/active_voice_studio_url.txt")

from pathlib import Path

# Read the URL saved by the existing Cloudflare tunnel code.
TUNNEL_URL = Path(
    "/kaggle/working/active_voice_studio_url.txt"
).read_text().strip()

print("\n" + "=" * 55)
print("🎙️ VOICESTUDIO IS READY")
print("=" * 55)

print("\n🌐 VoiceStudio URL:")
print(TUNNEL_URL)

print("\n🔑 API key:")
print(VOICESTUDIO_API_KEY)

print("\n⚡ Sign-in link:")
print(f"{TUNNEL_URL.rstrip('/')}/#api_key={VOICESTUDIO_API_KEY}")

print("\nKeep the URL and API key private.")

✅ Persistent cloudflared binary already exists.

🌐 Starting Cloudflare Quick Tunnel...
✅ Cloudflared execution permissions fixed.

🔥 VOICESTUDIO IS LIVE

Open this URL:
https://copied-nsw-aaa-avi.trycloudflare.com

Your VoiceStudio UI is now:
Kaggle T4 → VoiceStudio backend → Cloudflare → your browser

⚠️ Treat this URL as private.
Use VoiceStudio's share/security settings before sharing it.

URL saved to:
/kaggle/working/active_voice_studio_url.txt

🎙️ VOICESTUDIO IS READY

🌐 VoiceStudio URL:
https://copied-nsw-aaa-avi.trycloudflare.com

🔑 API key:
1r9iB0vsii3SmjBDTDZJqFKvq1p0p_NAxYXyTwutbGI

⚡ Sign-in link:
https://copied-nsw-aaa-avi.trycloudflare.com/#api_key=1r9iB0vsii3SmjBDTDZJqFKvq1p0p_NAxYXyTwutbGI

Keep the URL and API key private.


# 7

In [ ]:
# ── 7. Real TTS smoke test + persistent output ─────────────────────────────

import json
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

from IPython.display import Audio, display


BASE = "http://127.0.0.1:3900"

OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT = OUTPUT_DIR / "omnivoice_smoke.wav"


# -------------------------------------------------------------------
# Health check
# -------------------------------------------------------------------

with urllib.request.urlopen(
    f"{BASE}/health",
    timeout=10,
) as response:

    health_info = json.load(response)

print("GET /health ->")
print(json.dumps(health_info, indent=2))


# -------------------------------------------------------------------
# Generate real TTS
# -------------------------------------------------------------------

text = (
    "There are moments in technology when everything seems obvious "
    "until you look a little closer. "
    "And sometimes, the strangest stories begin there."
)

data = urllib.parse.urlencode(
    {
        "text": text,
    }
).encode()


print("\n🎙️ Generating speech on the T4...")

try:

    request = urllib.request.Request(
        f"{BASE}/generate",
        data=data,
    )

    with urllib.request.urlopen(
        request,
        timeout=1800,
    ) as response:

        wav = response.read()

        duration = response.headers.get("X-Audio-Duration")
        gen_time = response.headers.get("X-Gen-Time")
        seed = response.headers.get("X-Seed")

        print("\n✅ TTS GENERATION SUCCESS")

        print(f"Audio bytes: {len(wav):,}")
        print(f"Duration:    {duration}s")
        print(f"Generation:  {gen_time}s")
        print(f"Seed:        {seed}")

except urllib.error.HTTPError as e:

    detail = e.read().decode(
        "utf-8",
        errors="replace",
    )

    raise RuntimeError(
        f"Generation failed: HTTP {e.code}\n\n"
        f"{detail}\n\n"
        "Check /kaggle/working/omnivoice_backend.log"
    )


# -------------------------------------------------------------------
# Save permanently
# -------------------------------------------------------------------

with open(OUT, "wb") as f:
    f.write(wav)

print("\n💾 Saved permanently to:")
print(OUT)


# -------------------------------------------------------------------
# Play it
# -------------------------------------------------------------------

display(Audio(str(OUT)))


# -------------------------------------------------------------------
# Persistence summary
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("PERSISTENCE SUMMARY")
print("=" * 80)

print("Repository:")
print("/kaggle/working/VoiceStudio")

print("\nModel cache:")
print("/kaggle/working/omnivoice_cache")

print("\nVoiceStudio data:")
print("/kaggle/working/omnivoice_data")

print("\nGenerated notebook outputs:")
print("/kaggle/working/outputs")

print("\n✅ Everything above is under Kaggle's persistent working directory.")

In [13]:

from pathlib import Path
import shutil
import subprocess

source = Path("/kaggle/working/omnivoice_cache")
destination = Path("/tmp/omnivoice_cache_largefs")

if source.is_symlink():
    print(f"✅ Cache is already redirected to: {source.resolve()}")

elif not source.exists():
    raise FileNotFoundError(f"Cache directory not found: {source}")

elif destination.exists():
    raise FileExistsError(
        f"{destination} already exists. Nothing was changed; "
        "check the destination before proceeding."
    )

else:
    print("Moving the existing model cache to the larger filesystem...")
    print("This preserves existing cached files; it may take a while.")

    shutil.move(str(source), str(destination))
    source.symlink_to(destination, target_is_directory=True)

    print("✅ Cache moved and original path redirected.")

print(f"\nVoiceStudio cache path: {source}")
print(f"Actual cache location:  {source.resolve()}")
print(f"Free space for cache:   {shutil.disk_usage(source).free / 1024**3:.2f} GB")

print("\nFilesystem verification:")
subprocess.run(["df", "-h", str(source)], check=False)


Moving the existing model cache to the larger filesystem...
This preserves existing cached files; it may take a while.
✅ Cache moved and original path redirected.

VoiceStudio cache path: /kaggle/working/omnivoice_cache
Actual cache location:  /tmp/omnivoice_cache_largefs
Free space for cache:   1058.97 GB

Filesystem verification:
Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  7.0T  1.1T  88% /


CompletedProcess(args=['df', '-h', '/kaggle/working/omnivoice_cache'], returncode=0)